# HealthSync AI — Demand Forecasting, Stock-Out Detection & Smart Alerts

This notebook covers your three AI modules for the hackathon:

1. **Forecasting logic** — baseline model (7-day average + 14-day trend), predicting demand for the next 3/5/7 days
2. **Stock-out detection** — compares forecasted demand to current stock, calculates days left, applies a safety buffer, and produces a risk score
3. **Alert rules** — low stock, predicted stock-out, rapid consumption, and expiry alerts

**How to use this notebook:**
- Run all cells top to bottom. It works immediately on synthetic data so you can test/demo it right now.
- Section 8 shows how to swap in your teammate's real Postgres data once it's ready.
- Everything is driven by the `CONFIG` dict in Section 2 — tune thresholds there, not inside the functions.


## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)


## 2. Configuration

Everything tunable lives here. Adjust these numbers freely during the hackathon without touching the logic below.

In [ ]:
CONFIG = {
    "horizons": [3, 5, 7],              # forecast next N days (as asked: 3, 5, or 7)
    "recent_window_days": 7,             # baseline = average usage over last 7 days
    "trend_window_days": 14,             # trend = comparing first half vs second half of last 14 days
    "safety_buffer_days": 2,             # extra days of buffer added when scoring risk
    "low_stock_default_threshold": 20,   # fallback if a medicine has no reorder_threshold set
    "rapid_consumption_window_days": 3,  # "recent" window for spike detection
    "rapid_consumption_multiplier": 1.5, # recent avg >= 1.5x normal avg -> alert
    "expiry_warning_days": 30,           # flag anything expiring within 30 days
    "risk_thresholds": {"high": 0.7, "medium": 0.4},  # risk_score cutoffs
}


## 3. Data — synthetic generator (swap for real data later)

Expected shape once you're plugging in real data:

- **usage_df**: `medicine_id, date, quantity_used` — one row per medicine per day
- **medicine_df**: `medicine_id, medicine_name, current_stock, reorder_threshold, expiry_date`

The generator below fakes 30 days of usage across 6 medicines, each with a random baseline and trend, so you can run the whole pipeline end-to-end right now.

In [ ]:
def generate_synthetic_data(n_medicines=6, n_days=30, seed=42):
    rng = np.random.default_rng(seed)
    names = ['Paracetamol', 'Amoxicillin', 'ORS Sachets', 'Insulin', 'Iron Tablets', 'Cough Syrup']

    medicines = []
    for i, name in enumerate(names[:n_medicines]):
        medicines.append({
            'medicine_id': i + 1,
            'medicine_name': name,
            'current_stock': int(rng.integers(10, 200)),
            'reorder_threshold': int(rng.integers(15, 40)),
            'expiry_date': (datetime.now() + timedelta(days=int(rng.integers(5, 200)))).date()
        })
    medicine_df = pd.DataFrame(medicines)

    rows = []
    end_date = datetime.now().date()
    for med in medicines:
        base = rng.integers(5, 25)
        trend = rng.uniform(-0.3, 0.5)
        for d in range(n_days, 0, -1):
            date = end_date - timedelta(days=d)
            noise = rng.normal(0, base * 0.2)
            qty = max(0, round(base + trend * (n_days - d) + noise))
            rows.append({'medicine_id': med['medicine_id'], 'date': date, 'quantity_used': qty})
    usage_df = pd.DataFrame(rows)
    return usage_df, medicine_df

usage_df, medicine_df = generate_synthetic_data()
print(medicine_df)
print(usage_df.head())


## 4. Forecasting logic

Baseline model, exactly as scoped:
- **Baseline** = average usage over the last 7 days
- **Trend** = difference between the two halves of the last 14 days, turned into a daily slope
- **Forecast** = baseline + trend × day_offset, for each day out to the longest horizon, clipped at 0 (can't have negative demand)
- Cumulative demand is then computed for the 3/5/7-day horizons

In [ ]:
def compute_forecast(usage_df, medicine_id, as_of_date=None, config=CONFIG):
    df = usage_df[usage_df['medicine_id'] == medicine_id].copy()
    df['date'] = pd.to_datetime(df['date'])

    if as_of_date is None:
        as_of_date = df['date'].max()
    else:
        as_of_date = pd.to_datetime(as_of_date)

    df = df[df['date'] <= as_of_date].sort_values('date')

    recent_window = config['recent_window_days']
    trend_window = config['trend_window_days']

    recent = df[df['date'] > as_of_date - pd.Timedelta(days=recent_window)]
    baseline_avg = recent['quantity_used'].mean() if len(recent) else 0.0
    if pd.isna(baseline_avg):
        baseline_avg = 0.0

    trend_data = df[df['date'] > as_of_date - pd.Timedelta(days=trend_window)]
    daily_trend = 0.0
    if len(trend_data) >= trend_window:
        half = trend_window // 2
        first_half_avg = trend_data.iloc[:half]['quantity_used'].mean()
        second_half_avg = trend_data.iloc[half:]['quantity_used'].mean()
        daily_trend = (second_half_avg - first_half_avg) / half

    max_horizon = max(config['horizons'])
    daily_rows = []
    for day_offset in range(1, max_horizon + 1):
        predicted = max(baseline_avg + daily_trend * day_offset, 0.0)
        daily_rows.append({
            'medicine_id': medicine_id,
            'forecast_date': (as_of_date + pd.Timedelta(days=day_offset)).date(),
            'day_offset': day_offset,
            'predicted_usage': round(predicted, 2)
        })
    forecast_df = pd.DataFrame(daily_rows)

    cumulative = {}
    for h in config['horizons']:
        cumulative[h] = round(forecast_df[forecast_df['day_offset'] <= h]['predicted_usage'].sum(), 2)

    return forecast_df, cumulative, baseline_avg, daily_trend


# quick check on one medicine
f_df, cum, base, trend = compute_forecast(usage_df, medicine_id=1)
print("Baseline daily avg:", round(base, 2), "| Daily trend:", round(trend, 2))
print("Cumulative predicted demand by horizon:", cum)
f_df.head()


## 5. Stock-out detection

Turns the forecast into a decision: for each medicine, compute **days left** at the current demand rate, apply a **safety buffer**, and produce a 0–1 **risk score** plus a risk level (`LOW` / `MEDIUM` / `HIGH` / `CRITICAL`).

Risk score logic: the closer `days_left` is to the forecast horizon + buffer, the higher the score. If stock is already at zero, it's automatically `CRITICAL`.

In [ ]:
def detect_stockout_risk(medicine_id, current_stock, forecast_df, baseline_avg, config=CONFIG):
    max_horizon = max(config['horizons'])
    buffer = config['safety_buffer_days']

    avg_daily_demand = forecast_df['predicted_usage'].mean() if len(forecast_df) else baseline_avg
    if avg_daily_demand is None or pd.isna(avg_daily_demand):
        avg_daily_demand = 0.0

    if avg_daily_demand <= 0:
        days_left = float('inf')
    else:
        days_left = current_stock / avg_daily_demand

    effective_horizon = max_horizon + buffer

    if current_stock <= 0:
        risk_score, risk_level = 1.0, 'CRITICAL'
    else:
        if days_left == float('inf'):
            risk_score = 0.0
        else:
            risk_score = max(0.0, min(1.0, 1 - (days_left / effective_horizon)))

        thresholds = config['risk_thresholds']
        if risk_score >= thresholds['high']:
            risk_level = 'HIGH'
        elif risk_score >= thresholds['medium']:
            risk_level = 'MEDIUM'
        else:
            risk_level = 'LOW'

    return {
        'medicine_id': medicine_id,
        'current_stock': current_stock,
        'avg_daily_demand': round(avg_daily_demand, 2),
        'days_left': round(days_left, 1) if days_left != float('inf') else None,
        'risk_score': round(risk_score, 2),
        'risk_level': risk_level,
        'generated_at': datetime.now()
    }


# quick check
risk = detect_stockout_risk(1, medicine_df.iloc[0]['current_stock'], f_df, base)
risk


## 6. Alert rules

Four alert types, as scoped:
- **LOW_STOCK** — current stock at/below the reorder threshold (medicine-specific if set, else the default)
- **PREDICTED_STOCKOUT** — risk level from Section 5 is `HIGH` or `CRITICAL`
- **RAPID_CONSUMPTION** — recent (last 3 days) average usage spikes vs. the 14-day normal
- **EXPIRY** — expiry date within the warning window (only if the medicine has an expiry date)

Each alert crossing its threshold gets saved as a row so a dashboard can just query and display them.

In [ ]:
def generate_alerts(medicine_id, medicine_name, current_stock, reorder_threshold,
                     expiry_date, usage_df, risk_row, config=CONFIG):
    alerts = []
    now = datetime.now()

    # 1. Low stock alert
    threshold = reorder_threshold if reorder_threshold is not None else config['low_stock_default_threshold']
    if current_stock <= threshold:
        alerts.append({
            'medicine_id': medicine_id, 'medicine_name': medicine_name,
            'alert_type': 'LOW_STOCK',
            'severity': 'HIGH' if current_stock <= threshold * 0.5 else 'MEDIUM',
            'message': f"{medicine_name}: stock ({current_stock}) at/below reorder threshold ({threshold}).",
            'generated_at': now
        })

    # 2. Predicted stock-out alert
    if risk_row['risk_level'] in ('HIGH', 'CRITICAL'):
        days_txt = f"{risk_row['days_left']} days" if risk_row['days_left'] is not None else "unknown"
        alerts.append({
            'medicine_id': medicine_id, 'medicine_name': medicine_name,
            'alert_type': 'PREDICTED_STOCKOUT', 'severity': risk_row['risk_level'],
            'message': f"{medicine_name}: predicted to run out in ~{days_txt} at current usage rate.",
            'generated_at': now
        })

    # 3. Rapid consumption alert
    df = usage_df[usage_df['medicine_id'] == medicine_id].copy()
    df['date'] = pd.to_datetime(df['date'])
    last_date = df['date'].max()
    recent_win = config['rapid_consumption_window_days']
    recent_avg = df[df['date'] > last_date - pd.Timedelta(days=recent_win)]['quantity_used'].mean()
    normal_avg = df[df['date'] > last_date - pd.Timedelta(days=config['trend_window_days'])]['quantity_used'].mean()
    if normal_avg and recent_avg and normal_avg > 0 and recent_avg >= config['rapid_consumption_multiplier'] * normal_avg:
        alerts.append({
            'medicine_id': medicine_id, 'medicine_name': medicine_name,
            'alert_type': 'RAPID_CONSUMPTION', 'severity': 'MEDIUM',
            'message': f"{medicine_name}: usage over last {recent_win} days ({recent_avg:.1f}/day) is well above normal ({normal_avg:.1f}/day).",
            'generated_at': now
        })

    # 4. Expiry alert (optional field)
    if expiry_date is not None:
        expiry_ts = pd.to_datetime(expiry_date)
        days_to_expiry = (expiry_ts - pd.Timestamp(now.date())).days
        if days_to_expiry <= config['expiry_warning_days']:
            alerts.append({
                'medicine_id': medicine_id, 'medicine_name': medicine_name,
                'alert_type': 'EXPIRY', 'severity': 'HIGH' if days_to_expiry <= 7 else 'MEDIUM',
                'message': f"{medicine_name}: expires in {days_to_expiry} days ({expiry_ts.date()}).",
                'generated_at': now
            })

    return alerts


## 7. Full pipeline — run everything together

This is the single function you'll want to call from anywhere else (a script, an API endpoint, a scheduled job): give it usage data + medicine master data, get back three clean dataframes.

In [ ]:
def run_ai_pipeline(usage_df, medicine_df, config=CONFIG):
    all_forecasts, all_risks, all_alerts = [], [], []

    for _, med in medicine_df.iterrows():
        mid = med['medicine_id']

        f_df, cumulative, baseline_avg, trend = compute_forecast(usage_df, mid, config=config)
        f_df['medicine_name'] = med['medicine_name']
        all_forecasts.append(f_df)

        risk_row = detect_stockout_risk(mid, med['current_stock'], f_df, baseline_avg, config)
        risk_row['medicine_name'] = med['medicine_name']
        all_risks.append(risk_row)

        alerts = generate_alerts(
            mid, med['medicine_name'], med['current_stock'],
            med.get('reorder_threshold'), med.get('expiry_date'),
            usage_df, risk_row, config
        )
        all_alerts.extend(alerts)

    forecast_result_df = pd.concat(all_forecasts, ignore_index=True)
    risk_result_df = pd.DataFrame(all_risks)
    alerts_result_df = pd.DataFrame(all_alerts)
    return forecast_result_df, risk_result_df, alerts_result_df


forecast_result_df, risk_result_df, alerts_result_df = run_ai_pipeline(usage_df, medicine_df)

print("=== STOCK-OUT RISK ===")
display_cols = ['medicine_name', 'current_stock', 'avg_daily_demand', 'days_left', 'risk_score', 'risk_level']
print(risk_result_df[display_cols].sort_values('risk_score', ascending=False).to_string(index=False))

print("\n=== ALERTS ===")
if len(alerts_result_df):
    print(alerts_result_df[['medicine_name', 'alert_type', 'severity', 'message']].to_string(index=False))
else:
    print("No alerts triggered.")


## 8. Quick visualization (handy for your demo/PPT)

In [ ]:
def plot_medicine_forecast(medicine_id, usage_df, forecast_df, medicine_df):
    name = medicine_df[medicine_df['medicine_id'] == medicine_id]['medicine_name'].iloc[0]
    hist = usage_df[usage_df['medicine_id'] == medicine_id].copy()
    hist['date'] = pd.to_datetime(hist['date'])
    fc = forecast_df[forecast_df['medicine_id'] == medicine_id].copy()
    fc['forecast_date'] = pd.to_datetime(fc['forecast_date'])

    plt.figure(figsize=(9, 4))
    plt.plot(hist['date'], hist['quantity_used'], label='Actual usage', marker='o', markersize=3)
    plt.plot(fc['forecast_date'], fc['predicted_usage'], label='Forecast', marker='o', markersize=3, linestyle='--')
    plt.axvline(hist['date'].max(), color='gray', linestyle=':', linewidth=1)
    plt.title(f"Usage & Forecast — {name}")
    plt.xlabel("Date")
    plt.ylabel("Units used")
    plt.legend()
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

plot_medicine_forecast(1, usage_df, forecast_result_df, medicine_df)


## 9. Connecting to the real database

Once Member 2's Postgres schema is up, you have two options. **Option A is simplest for a hackathon** since your pipeline already returns clean dataframes.

### Option A — read/write Postgres directly from this notebook (or a scheduled script)

```python
# !pip install psycopg2-binary sqlalchemy
from sqlalchemy import create_engine

DB_CONFIG = {
    "host": "localhost",       # or your hosted DB host
    "port": 5432,
    "dbname": "healthsync",
    "user": "postgres",
    "password": "your_password"
}

def get_engine():
    url = (f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
           f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}")
    return create_engine(url)

engine = get_engine()

# Pull real data instead of the synthetic generator:
# usage_df = pd.read_sql("SELECT medicine_id, date, quantity_used FROM medicine_usage_log", engine)
# medicine_df = pd.read_sql(
#     "SELECT id AS medicine_id, name AS medicine_name, current_stock, reorder_threshold, expiry_date FROM medicine",
#     engine
# )

# Write results back so the dashboard (Spring Boot) can just query these tables:
# forecast_result_df.to_sql('forecast_results', engine, if_exists='append', index=False)
# risk_result_df.to_sql('stockout_risk', engine, if_exists='append', index=False)
# alerts_result_df.to_sql('alerts', engine, if_exists='append', index=False)
```

Suggested tables to ask Member 2 to create (or create yourself with `CREATE TABLE`):

```sql
CREATE TABLE forecast_results (
    id SERIAL PRIMARY KEY,
    medicine_id INT NOT NULL,
    medicine_name VARCHAR(255),
    forecast_date DATE NOT NULL,
    day_offset INT NOT NULL,
    predicted_usage NUMERIC,
    generated_at TIMESTAMP DEFAULT NOW()
);

CREATE TABLE stockout_risk (
    id SERIAL PRIMARY KEY,
    medicine_id INT NOT NULL,
    medicine_name VARCHAR(255),
    current_stock INT,
    avg_daily_demand NUMERIC,
    days_left NUMERIC,
    risk_score NUMERIC,
    risk_level VARCHAR(20),
    generated_at TIMESTAMP DEFAULT NOW()
);

CREATE TABLE alerts (
    id SERIAL PRIMARY KEY,
    medicine_id INT NOT NULL,
    medicine_name VARCHAR(255),
    alert_type VARCHAR(50),
    severity VARCHAR(20),
    message TEXT,
    generated_at TIMESTAMP DEFAULT NOW()
);
```

### Option B — expose this as a small REST API instead

If it's easier for the Spring Boot backend to *call* your AI logic rather than share a DB connection, wrap `run_ai_pipeline()` in a tiny FastAPI service (`/forecast`, `/risk`, `/alerts` endpoints) and have Spring Boot hit it over HTTP. This is a bit more setup than Option A, so only go this route if direct DB access is inconvenient for your team's split of work.

### One important note about Colab specifically
Colab notebooks are great for *building and testing* this logic (which is what this notebook does), but they aren't meant to run as an always-on service for your final demo — the runtime disconnects and isn't reachable from your backend. For the actual integrated app, plan to move this same code into a `.py` script (or the FastAPI wrapper above) that runs alongside your Spring Boot backend, or into a scheduled job that writes to Postgres. Colab stays perfect for prototyping the model and generating the charts for your demo/PPT.
